In [43]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/nerocaesarsuprobo/dataset-spotify/dataset_spotify_sentimen.csv


In [44]:
# Install library Sastrawi untuk NLP Bahasa Indonesia
!pip install Sastrawi

In [45]:
import pandas as pd
import re
import string
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

In [46]:
file_path = '/kaggle/input/datasets/nerocaesarsuprobo/dataset-spotify/dataset_spotify_sentimen.csv' 

df = pd.read_csv(file_path)

print("Informasi Dataset Awal:")
print(df.info())
print("\nSample Data:")
display(df.head())

Informasi Dataset Awal:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11000 entries, 0 to 10999
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   content  11000 non-null  object
 1   score    11000 non-null  int64 
 2   label    11000 non-null  object
dtypes: int64(1), object(2)
memory usage: 257.9+ KB
None

Sample Data:


,content,score,label
0,aplikasi ini bagus untuk lagu rohani Kristen t...,5,positif
1,kikir,3,netral
2,oke aku senang denger lagu lagunya tp iklannya...,5,positif
3,banyak iklaannnn,5,positif
4,"memilih lagu untuk diputar masih ada limit,jik...",2,negatif


In [47]:
# Kamus Normalisasi (Tambahkan kata slang lain jika dirasa perlu)
norm_dict = {
    "yg": "yang", "gk": "tidak", "gak": "tidak", "tdk": "tidak",
    "bgt": "banget", "kl": "kalau", "kalo": "kalau", "udh": "sudah",
    "udah": "sudah", "dgn": "dengan", "tp": "tapi", "aplikasinya": "aplikasi",
    "premium": "premium", "spotify": "spotify", "ads": "iklan", "ad": "iklan",
    "error": "galat", "bug": "galat", "lemot": "lambat", "lemotnya": "lambat",
    "bagus": "baik", "bgs": "baik", "keren": "bagus", "mantap": "bagus"
}

def clean_text(text):
    # Handle missing values (jika ada baris kosong)
    if not isinstance(text, str):
        return ""
        
    # 1. Case Folding
    text = text.lower()
    
    # 2. Hapus URL, Username (@), dan Hashtag (#)
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'@\w+|#\w+', '', text)
    
    # 3. Hapus tanda baca dan angka (sisakan huruf saja)
    text = text.translate(str.maketrans('', '', string.punctuation + string.digits))
    
    # 4. Normalisasi kata slang
    words = text.split()
    normalized_words = [norm_dict.get(w, w) for w in words]
    text = ' '.join(normalized_words)
    
    # 5. Hapus spasi berlebih
    text = text.strip()
    return text

In [48]:
print("Memulai proses pembersihan teks...")

# Terapkan fungsi clean_text ke kolom 'content'
df['clean_content'] = df['content'].apply(clean_text)

# Hapus baris yang teksnya menjadi kosong setelah dibersihkan
df = df[df['clean_content'] != ""]

print("Proses pembersihan selesai. Sample data bersih:")
display(df[['content', 'clean_content']].head())

Memulai proses pembersihan teks...
Proses pembersihan selesai. Sample data bersih:


,content,clean_content
0,aplikasi ini bagus untuk lagu rohani Kristen t...,aplikasi ini baik untuk lagu rohani kristen te...
1,kikir,kikir
2,oke aku senang denger lagu lagunya tp iklannya...,oke aku senang denger lagu lagunya tapi iklann...
3,banyak iklaannnn,banyak iklaannnn
4,"memilih lagu untuk diputar masih ada limit,jik...",memilih lagu untuk diputar masih ada limitjika...


In [49]:
print("Memulai penghapusan stopwords...")

# Inisiasi Sastrawi Stopword Remover
factory = StopWordRemoverFactory()
stopword_remover = factory.create_stop_word_remover()

def remove_stopwords(text):
    return stopword_remover.remove(text)

# Terapkan penghapusan stopwords
df['final_content'] = df['clean_content'].apply(remove_stopwords)

# Hapus baris kosong (lagi) untuk memastikan data aman
df = df[df['final_content'].str.strip() != ""]
df.dropna(subset=['final_content', 'label'], inplace=True)

print("Penghapusan stopwords selesai. Sample hasil akhir:")
display(df[['clean_content', 'final_content', 'label']].head())

Memulai penghapusan stopwords...
Penghapusan stopwords selesai. Sample hasil akhir:


,clean_content,final_content,label
0,aplikasi ini baik untuk lagu rohani kristen te...,aplikasi baik lagu rohani kristen terbaru,positif
1,kikir,kikir,netral
2,oke aku senang denger lagu lagunya tapi iklann...,oke aku senang denger lagu lagunya iklannya jg...,positif
3,banyak iklaannnn,banyak iklaannnn,positif
4,memilih lagu untuk diputar masih ada limitjika...,memilih lagu diputar ada limitjika mencapai li...,negatif


In [50]:
# Cek distribusi kelas setelah data kotor dibuang
print("Distribusi Kelas (Label) Saat Ini:")
print(df['label'].value_counts())

# Simpan ke CSV untuk dipakai di tahap modeling
output_file = 'preprocessed_spotify_data.csv'
df.to_csv(output_file, index=False)

print(f"\nData bersih berhasil disimpan sebagai '{output_file}'!")
print(f"Total data siap latih: {len(df)} baris.")

Distribusi Kelas (Label) Saat Ini:
label
positif    6242
negatif    4105
netral      570
Name: count, dtype: int64

Data bersih berhasil disimpan sebagai 'preprocessed_spotify_data.csv'!
Total data siap latih: 10917 baris.


In [51]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, GRU, Dense, SpatialDropout1D, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

print("TensorFlow version:", tf.__version__)
# Cek apakah GPU terdeteksi
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

TensorFlow version: 2.19.0
Num GPUs Available:  1


In [52]:
# Load data yang sudah bersih dari tahap sebelumnya
df = pd.read_csv('preprocessed_spotify_data.csv')

# Encode label (negatif -> 0, netral -> 1, positif -> 2)
le = LabelEncoder()
df['label_encoded'] = le.fit_transform(df['label'])

X = df['final_content'].astype(str)
y = df['label_encoded']

print("Mapping Label:", dict(zip(le.classes_, le.transform(le.classes_))))

Mapping Label: {'negatif': np.int64(0), 'netral': np.int64(1), 'positif': np.int64(2)}


In [53]:
# Konfigurasi Tokenizer
max_features = 5000  # Maksimal kosakata yang diambil
max_len = 100        # Panjang maksimal tiap kalimat (dipotong/ditambah padding jika kurang)

tokenizer = Tokenizer(num_words=max_features, oov_token="<OOV>")
tokenizer.fit_on_texts(X)

X_seq = tokenizer.texts_to_sequences(X)
X_pad = pad_sequences(X_seq, maxlen=max_len, padding='post', truncating='post')

print("Dimensi data X setelah padding:", X_pad.shape)

Dimensi data X setelah padding: (10917, 100)


In [54]:
callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-5)
]

In [55]:
from tensorflow.keras.layers import GlobalMaxPooling1D

print("--- SKEMA 1: LSTM (80/20 Split) DENGAN MASKING ---")
# Pembagian data 80% training, 20% testing
X_train_1, X_test_1, y_train_1, y_test_1 = train_test_split(X_pad, y, test_size=0.20, random_state=42)

model_1 = Sequential([
    # mask_zero=True agar model tidak amnesia dengan angka 0 dari padding
    Embedding(input_dim=max_features, output_dim=128, mask_zero=True),
    
    # return_sequences=True wajib agar outputnya bisa diteruskan ke GlobalMaxPooling1D
    LSTM(64, return_sequences=True, dropout=0.2), 
    
    GlobalMaxPooling1D(), # Radar pencari kata kunci sentimen
    
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(3, activation='softmax')
])

model_1.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

history_1 = model_1.fit(
    X_train_1, y_train_1, 
    epochs=10, 
    batch_size=64, # Batch size disamakan agar adil dan cepat belajar
    validation_data=(X_test_1, y_test_1), 
    callbacks=callbacks,
    verbose=1
)

# Evaluasi
loss_1, acc_1 = model_1.evaluate(X_test_1, y_test_1, verbose=0)
print(f"\nAkurasi Testing Skema 1 (LSTM 80/20): {acc_1 * 100:.2f}%")

--- SKEMA 1: LSTM (80/20 Split) DENGAN MASKING ---
Epoch 1/10


/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:965: UserWarning: Layer 'global_max_pooling1d' (of type GlobalMaxPooling1D) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


  1/137 ━━━━━━━━━━━━━━━━━━━━ 4:41 2s/step - accuracy: 0.5781 - loss: 1.0944

I0000 00:00:1777289830.728367     138 cuda_dnn.cc:529] Loaded cuDNN version 91002


137/137 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.7091 - loss: 0.8248 - val_accuracy: 0.8718 - val_loss: 0.4280 - learning_rate: 0.0010
Epoch 2/10
137/137 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.8793 - loss: 0.4032 - val_accuracy: 0.8764 - val_loss: 0.4040 - learning_rate: 0.0010
Epoch 3/10
137/137 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.8984 - loss: 0.3336 - val_accuracy: 0.8718 - val_loss: 0.4077 - learning_rate: 0.0010
Epoch 4/10
137/137 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9071 - loss: 0.2955 - val_accuracy: 0.8686 - val_loss: 0.4233 - learning_rate: 0.0010
Epoch 5/10
137/137 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9190 - loss: 0.2518 - val_accuracy: 0.8672 - val_loss: 0.4415 - learning_rate: 5.0000e-04

Akurasi Testing Skema 1 (LSTM 80/20): 87.64%


In [56]:
from tensorflow.keras.layers import GRU

print("--- SKEMA 2: GRU (80/20 Split) DENGAN MASKING ---")
# Data split menggunakan variabel yang sama dari skema 1 (sudah 80/20)

model_2 = Sequential([
    Embedding(input_dim=max_features, output_dim=128, mask_zero=True),
    
    # Menggunakan GRU sebagai variasi algoritma (syarat Dicoding)
    GRU(64, return_sequences=True, dropout=0.2),
    
    GlobalMaxPooling1D(),
    
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(3, activation='softmax')
])

model_2.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

history_2 = model_2.fit(
    X_train_1, y_train_1, 
    epochs=10, 
    batch_size=64, 
    validation_data=(X_test_1, y_test_1), 
    callbacks=callbacks,
    verbose=1
)

# Evaluasi
loss_2, acc_2 = model_2.evaluate(X_test_1, y_test_1, verbose=0)
print(f"\nAkurasi Testing Skema 2 (GRU 80/20): {acc_2 * 100:.2f}%")

--- SKEMA 2: GRU (80/20 Split) DENGAN MASKING ---
Epoch 1/10


/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:965: UserWarning: Layer 'global_max_pooling1d_1' (of type GlobalMaxPooling1D) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


137/137 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.6876 - loss: 0.8159 - val_accuracy: 0.8709 - val_loss: 0.4189 - learning_rate: 0.0010
Epoch 2/10
137/137 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.8828 - loss: 0.3827 - val_accuracy: 0.8745 - val_loss: 0.4025 - learning_rate: 0.0010
Epoch 3/10
137/137 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.8908 - loss: 0.3378 - val_accuracy: 0.8800 - val_loss: 0.4052 - learning_rate: 0.0010
Epoch 4/10
137/137 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9079 - loss: 0.2780 - val_accuracy: 0.8649 - val_loss: 0.4341 - learning_rate: 0.0010
Epoch 5/10
137/137 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9228 - loss: 0.2514 - val_accuracy: 0.8636 - val_loss: 0.4643 - learning_rate: 5.0000e-04
Epoch 6/10
137/137 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9368 - loss: 0.2010 - val_accuracy: 0.8558 - val_loss: 0.4802 - learning_rate: 5.0000e-04

Akurasi Testing Skema 2 (GRU 80/20): 88.00%


In [57]:
from tensorflow.keras.layers import Bidirectional, GlobalMaxPooling1D

print("--- SKEMA 3: BIDIRECTIONAL LSTM (Target > 92%) ---")
X_train_3, X_test_3, y_train_3, y_test_3 = train_test_split(X_pad, y, test_size=0.10, random_state=42)

model_3 = Sequential([
    # mask_zero=True tetap dipakai agar tidak amnesia sama padding
    Embedding(input_dim=max_features, output_dim=128, mask_zero=True),
    
    # Bidirectional: Membaca teks dari dua arah (Kiri->Kanan & Kanan->Kiri)
    # return_sequences=True wajib dipakai kalau mau disambung ke GlobalMaxPooling
    Bidirectional(LSTM(64, return_sequences=True, dropout=0.2)),
    
    # GlobalMaxPooling1D: Mengambil sinyal sentimen paling kuat dari seluruh token
    GlobalMaxPooling1D(),
    
    Dense(64, activation='relu'),
    Dropout(0.2), # Dropout dikecilin sedikit biar lebih banyak informasi yang lolos
    Dense(3, activation='softmax')
])

model_3.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# Training tanpa class_weight untuk fokus mengejar pure Accuracy
history_3 = model_3.fit(
    X_train_3, y_train_3, 
    epochs=10, 
    batch_size=64, # Batch size diperkecil agar akurasi lebih cepat naik
    validation_data=(X_test_3, y_test_3), 
    callbacks=callbacks,
    verbose=1
)

# Evaluasi Akhir
loss_train, acc_train = model_3.evaluate(X_train_3, y_train_3, verbose=0)
loss_test, acc_test = model_3.evaluate(X_test_3, y_test_3, verbose=0)

print("\n=== HASIL AKURASI SKEMA 3 TERBARU ===")
print(f"Akurasi Training : {acc_train * 100:.2f}%")
print(f"Akurasi Testing  : {acc_test * 100:.2f}%")

--- SKEMA 3: BIDIRECTIONAL LSTM (Target > 92%) ---
Epoch 1/10


/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:965: UserWarning: Layer 'global_max_pooling1d_2' (of type GlobalMaxPooling1D) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


154/154 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - accuracy: 0.7020 - loss: 0.8025 - val_accuracy: 0.8672 - val_loss: 0.4357 - learning_rate: 0.0010
Epoch 2/10
154/154 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.8824 - loss: 0.3784 - val_accuracy: 0.8608 - val_loss: 0.4256 - learning_rate: 0.0010
Epoch 3/10
154/154 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.8986 - loss: 0.3208 - val_accuracy: 0.8608 - val_loss: 0.4532 - learning_rate: 0.0010
Epoch 4/10
154/154 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.9173 - loss: 0.2585 - val_accuracy: 0.8690 - val_loss: 0.4542 - learning_rate: 0.0010
Epoch 5/10
154/154 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.9344 - loss: 0.2143 - val_accuracy: 0.8562 - val_loss: 0.5369 - learning_rate: 5.0000e-04
Epoch 6/10
154/154 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.9390 - loss: 0.1997 - val_accuracy: 0.8361 - val_loss: 0.5811 - learning_rate: 5.0000e-04
Epoch 7/10
154/154 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.9502 - loss: 0.175

In [58]:
print("=== REKAPITULASI AKURASI TESTING ===")
print(f"Skema 1 (LSTM, 80/20): {acc_1 * 100:.2f}%")
print(f"Skema 2 (GRU, 80/20) : {acc_2 * 100:.2f}%")
print(f"Skema 3 (LSTM, 90/10): {acc_3 * 100:.2f}%")

# Menampilkan Classification Report dari model terbaik (misal model_3)
y_pred_3 = np.argmax(model_3.predict(X_test_3), axis=-1)
print("\nClassification Report (Skema 3):")
print(classification_report(y_test_3, y_pred_3, target_names=le.classes_))

=== REKAPITULASI AKURASI TESTING ===
Skema 1 (LSTM, 80/20): 87.64%
Skema 2 (GRU, 80/20) : 88.00%
Skema 3 (LSTM, 90/10): 58.33%
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step

Classification Report (Skema 3):
              precision    recall  f1-score   support

     negatif       0.82      0.91      0.86       407
      netral       0.00      0.00      0.00        48
     positif       0.91      0.91      0.91       637

    accuracy                           0.87      1092
   macro avg       0.58      0.61      0.59      1092
weighted avg       0.84      0.87      0.85      1092



In [59]:
def prediksi_sentimen(teks):
    # 1. Terapkan Preprocessing yang SAMA PERSIS dengan data training
    teks_bersih = clean_text(teks)
    teks_final = remove_stopwords(teks_bersih)
    
    # Cegah error jika teks jadi kosong setelah dibersihkan
    if teks_final == "":
        return f"Teks: '{teks}'\nPrediksi: GAGAL (Teks tidak memiliki makna setelah dibersihkan)\n"
    
    # 2. Tokenisasi dan Padding
    sequence = tokenizer.texts_to_sequences([teks_final])
    padded = pad_sequences(sequence, maxlen=max_len, padding='post', truncating='post')
    
    # 3. Prediksi dengan model
    prediksi = model_3.predict(padded, verbose=0)
    kelas_prediksi = np.argmax(prediksi, axis=-1)[0]
    
    # 4. Ambil label dan probabilitas
    label_hasil = le.inverse_transform([kelas_prediksi])[0]
    probabilitas = np.max(prediksi) * 100
    
    # Return hasil dengan menampilkan teks yang sudah diproses
    return f"Teks Asli   : '{teks}'\nTeks Bersih : '{teks_final}'\nPrediksi    : {label_hasil.upper()} (Kepercayaan: {probabilitas:.2f}%)\n"

# ==========================================
# Uji coba Inference dengan kalimat baru
# ==========================================
print("=== TESTING INFERENCE ===")
print(prediksi_sentimen("Aplikasinya sering banget galat pas muter lagu, kecewa berat."))
print(prediksi_sentimen("Layanannya standar aja sih, gak ada yang spesial tapi bisa dipake."))
print(prediksi_sentimen("Keren banget fitur ai terbarunya, bikin playlist jadi gampang!"))

=== TESTING INFERENCE ===
Teks Asli   : 'Aplikasinya sering banget galat pas muter lagu, kecewa berat.'
Teks Bersih : 'aplikasi sering banget galat pas muter lagu kecewa berat'
Prediksi    : NEGATIF (Kepercayaan: 85.06%)

Teks Asli   : 'Layanannya standar aja sih, gak ada yang spesial tapi bisa dipake.'
Teks Bersih : 'layanannya standar aja sih ada spesial bisa dipake'
Prediksi    : POSITIF (Kepercayaan: 89.23%)

Teks Asli   : 'Keren banget fitur ai terbarunya, bikin playlist jadi gampang!'
Teks Bersih : 'bagus banget fitur ai terbarunya bikin playlist jadi gampang'
Prediksi    : POSITIF (Kepercayaan: 80.99%)

